In [1]:
import json
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

In [4]:
client = Elasticsearch("http://localhost:9200")


# import warnings
# from elasticsearch.exceptions import ElasticsearchWarning

# # To hide messages for HTTPS
# warnings.simplefilter('ignore', ElasticsearchWarning)
# warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# # Connection to Elasticsearch 8.x
# client = Elasticsearch(
#     "https://localhost:9200",
#     basic_auth=("elastic", "lcruhVaFiAjkVXltXOV2"),
#     verify_certs=False
# )

# # # Confirm connection
# print(client.info())

In [6]:
all_documents = []

file_path = "IR2026/documents.jsonl"

# Open jsonl file for reading
with open(file_path, "r", encoding="utf-8") as file:
    for line in file:
        # Convert each line from JSON string to a Python dictionary
        json_data = json.loads(line)
        all_documents.append(json_data)

print(f"{len(all_documents)} Successfully read!")

if len(all_documents) > 0:
    print("\nFirst document's structure:")
    print(all_documents[0])

FileNotFoundError: [Errno 2] No such file or directory: 'IR2026/documents.jsonl'

In [ ]:
index_name = "ir2026_index"

# Delete the index if it already exists
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

# Define Stucture (Mapping & Settings)
mapping = {
    "settings": {
        "number_of_shards": 1,
        "index": {
            "similarity": {
                "default": {
                    "type": "BM25",
                    "k1": 1.5,  # Μπορείς να πειράξεις το k1 άφοβα στα settings
                    "b": 0.75   # Και το b
                }
            }
        },
        "analysis": {
            "analyzer": {
                "default": {
                    "type": "english"
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "title": {"type": "text"},
            "text": {"type": "text"}
        }
    }
}

# Create Index 
client.indices.create(index=index_name, body=mapping)
print(f"The index '{index_name}' was created.")

# Prepare data for Bulk Insert
bulk_documents = []
for doc in all_documents:
    bulk_documents.append({
        "_index": index_name,
        "_id": doc["_id"],  #Keep the initial ID to match with the qrels
        "_source": {
            "title": doc["title"],
            "text": doc["text"]
        }
    })


# Bulk Insert Of the documents to ElasticSearch
print("Data Insert is starting. Please wait...")
bulk(client, bulk_documents, refresh=True)

# Confirmation
count = client.cat.count(index=index_name, format="json")[0]["count"]
print(f"The Insert completed! The index has {count} documents.")

Το ευρετήριο 'ir2026_index' δημιουργήθηκε.
Data Insert is starting. Please wait...
The Insert completed! The index has 40221 documents.


In [ ]:
# Load Queries
queries = []
with open("IR2026/queries.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        queries.append(json.loads(line))
        
print(f"Loaded {len(queries)} queries. Starting search...")

#Execute the search for k = 20, 30, 50

for k in [20, 30, 50]:
    results = {}
    for q in queries:
        q_id = str(q["_id"])
        q_text = q["text"]
        
        response = client.search(
            index=index_name, 
            size=k,
            query={
                "multi_match": {
                    "query": q_text,
                    "fields": ["title", "text"]
                }}
        )
        
#Export only the document IDs
        returned_ids = []
        for hit in response["hits"]["hits"]:
            returned_ids.append(hit["_id"])
            
        results[q_id] = returned_ids
        
#Save the results to a JSON file
    filename = f"results_k{k}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"Successfully saved results for k={k} in {filename}")

In [ ]:
for k in [20, 30, 50]:
    input_json = f"results_k{k}.json"
    output_trec = f"run_k{k}.txt"
    try:
        with open(input_json, "r", encoding="utf-8") as f:
            results = json.load(f)    
        # Create the TREC format file for the specific k
        with open(output_trec, "w", encoding="utf-8") as out_f:
            for q_id, doc_ids in results.items():
                # enumerate() gives us the rank of the document, starting from 1
                for rank, doc_id in enumerate(doc_ids, start=1):
                    # trec_eval mainly cares about the rank, so we create a synthetic score
                    score = 1.0 / rank 
                    trec_line = f"{q_id} Q0 {doc_id} {rank} {score:.4f} run_{k}\n"
                    out_f.write(trec_line)
        print(f"Success: Created evaluation file {output_trec}")
    except FileNotFoundError:
        print(f"Error: File {input_json} not found. Ensure it was created in the previous step.")